# Dataset analysis

## Configuration

In [ ]:
import sys
sys.path.append('../src')

Load machine specific configuration

In [ ]:
CONFIGURATION = '../configuration'

import configparser
import os
conf = configparser.ConfigParser()
conf.read((
    os.path.join(CONFIGURATION, f"default.conf"),
    os.path.join(CONFIGURATION, f"{os.uname().nodename}.conf")
))
conf.sections()

## Data Import

Load list of token / participants

In [ ]:
FILE_TOKENS = os.path.join(
    conf['Dataset']['path-root'],
    conf['Dataset']['file-token-list']
)
import pandas as pd
tokens = set(pd.read_pickle(FILE_TOKENS))

print(len(tokens))

Load conditions

In [ ]:
DIR_WINDOWS = os.path.join(
    conf['Dataset']['path-root'],
    conf['Dataset']['path-windows']
)
conditions = set(conf['Dataset']['conditions'].split(', '))
assert len(conditions - set(os.listdir(DIR_WINDOWS))) == 0
conditions

Load states

In [ ]:
states = set(conf['Dataset']['states'].split(', '))

assert len(states - {
    state
    for condition in conditions
    for state in os.listdir(os.path.join(DIR_WINDOWS, condition))
}) == 0
states


Create Data Loader

In [ ]:
from dataloader import LazyCachedDataLoader
dl = LazyCachedDataLoader(DIR_WINDOWS, conditions, states, tokens)

In [ ]:
from utils.csv import csv_col_dict
ACTION_UNITS = csv_col_dict(
    conf['Features']['action-units'].strip().split('\n')
)

## Quantitative analysis

In [ ]:
import matplotlib.pyplot as plt

### Distribution

In [ ]:
data = dl.get_full_set()
ax = data[ACTION_UNITS['feature']].plot(kind='box', showfliers=False)
ax.set_xticklabels(ACTION_UNITS['name'], rotation=90)
plt.title('Overall AU expression distribution')
plt.show()

=> AU28 Trivial because no variance

In [ ]:
def plot_cross_domains(title:str, plot, figsize=(10,6)):
    _, axs = plt.subplots(2,2, figsize=figsize, sharex=True, sharey=True, tight_layout=True)
    for i, (c, ax) in enumerate(zip(('neutral', 'stress'), axs)):
        ax[0].set_ylabel(c)
        for s, a in zip(('understanding', 'confusion'), ax):
            plot(a, c, s)
            if i == 1:
                a.set_xlabel(s)
                a.tick_params(axis='x', rotation=90)

    plt.suptitle(title)
    plt.show()

def plot_distribution(ax, condition, state):
    data = dl.get_domain(condition, state)[ACTION_UNITS['feature']]
    ax = data.plot(kind='box', ax=ax, showfliers=False)
    ax.set_xticks(range(1,len(ACTION_UNITS['name'])+1),
                  ACTION_UNITS['name'], rotation=90)

plot_cross_domains('AU expression distribution per domain', plot_distribution)

=> Show similar distribution patterns through domains

### Variances

In [ ]:
data = dl.get_full_set()
ax = data[ACTION_UNITS['feature']].var().plot(
    kind='bar', label='variance of full dataset'
)
ax = data.groupby('token')[ACTION_UNITS['feature']].var().mean().plot(
    kind='bar', ax=ax, label='mean of variances per participant',
    color='violet', alpha=.75
)
ax.set_xticklabels(ACTION_UNITS['name'], rotation=90)
plt.title('Overall AU expression variance')
plt.legend()
plt.show()

=> Lower variance per participant.

In [ ]:
ax = data.groupby('token')[ACTION_UNITS['feature']].var().plot(kind='box')
ax.set_xticklabels(ACTION_UNITS['name'], rotation=90)
plt.title('Variance per participant distribution')
plt.show()

=> 

In [ ]:
def plot_variances(ax, condition, state):
    data = dl.get_full_set()
    ax = data[ACTION_UNITS['feature']].var().plot(
        kind='bar', ax=ax, label='(reference) full set',
        color='darkgray', width=0.1
    )
    data = dl.get_domain(condition, state)
    ax = data[ACTION_UNITS['feature']].var().plot(
        kind='bar', ax=ax, label='total domain',
        alpha=0.75, 
    )
    ax = data.groupby('token')[ACTION_UNITS['feature']].var().mean().plot(
        kind='bar', ax=ax, label='mean per participant',
        color='violet', alpha=.75
    )
    ax.set_xticks(range(1,len(ACTION_UNITS['name'])+1),
                  ACTION_UNITS['name'], rotation=90)
    ax.legend()

plot_cross_domains('AU expression variance per domain', plot_variances)

=> In domain variance usually lower than cross domain variance;
Variance per participant lower than overall variance

=> Individual participant characteristics (per domain)

**Cross domain variances**
As foundation we take the mean of variances per participant denoted as $var_p$.
Action unit set is denotet by $x_{nu}$ with **n**eutral condition and **u**nderstanding state.

Cros state $mean(var_p(x_{nu} \cup x_{nc}), var_p(x_{su} \cup x_{sc}))$

Cros condition $mean(var_p(x_{nu} \cup x_{su}), var_p(x_{nc} \cup x_{sc}))$

Total $var_p(x_{nu} \cup x_{nc} \cup x_{su} \cup x_{sc})$

Individual $mean(var_p(x_{nu}), var_p(x_{nc}), var_p(x_{su}), var_p(x_{sc}))$

In [ ]:
def var_p(df:pd.DataFrame) -> pd.DataFrame:
    return df.groupby('token')[ACTION_UNITS['feature']].var().mean()
    
data_nu = dl.get_domain('neutral', 'understanding')
data_su = dl.get_domain('stress', 'understanding')
data_nc = dl.get_domain('neutral', 'confusion')
data_sc = dl.get_domain('stress', 'confusion')

total = var_p(pd.concat((data_nu, data_nc, data_su, data_sc)))
individual = pd.concat(
    (var_p(data_nu), var_p(data_nc), var_p(data_su), var_p(data_su)),
    axis=1
).T.mean()
x_state = pd.concat(
    (var_p(pd.concat((data_nu, data_nc))), var_p(pd.concat((data_su, data_sc)))),
    axis=1
).T.mean()
x_condi = pd.concat(
    (var_p(pd.concat((data_nu, data_su))), var_p(pd.concat((data_nc, data_sc)))),
    axis=1
).T.mean()

data = pd.concat((total, x_state, x_condi, individual), axis=1)
data.columns = ('total', 'x-state', 'x-condi', 'individual')

ax = data.plot(kind='bar')
ax.set_xticklabels(ACTION_UNITS['name'], rotation=90)
plt.title('Mean variance per participant across domains')
plt.show()




=> All cross variances below total (significance test?)

=> indicator for domain specific expressions

=> wheter state or condision domain has lower variance varies throughout action units

=> Run correlation analysis to find prominet AUs

### Correlation

In [ ]:
def plot_corr(ax, condition, state):
    data = dl.get_domain(condition, state)[ACTION_UNITS['feature']].corr(method='pearson')
    ax.matshow(data, cmap='coolwarm', vmin=-1, vmax=1)
    ax.set_xticks(range(len(ACTION_UNITS['name'])), ACTION_UNITS['name'])
    ax.set_yticks(range(len(ACTION_UNITS['name'])), ACTION_UNITS['name'])

plot_cross_domains('In-Domain Action Unit correlation', plot_corr, figsize=(7,7))

=> Overall similar correlation pattern

In [ ]:
_, axs = plt.subplots(2,2, figsize=(10,6), sharex=True, sharey=True, tight_layout=True)
rows = (('neutral', 'confusion'), ('stress', 'understanding'))
cols = (('neutral', 'understanding'), ('stress', 'confusion'))
for i, ((c_y,s_y), ax) in enumerate(zip(rows, axs)):
    ax[0].set_ylabel(f"{c_y}+{s_y}")
    for (c_x, s_x), a in zip(cols, ax):
        data_y = dl.get_domain(c_y, s_y).groupby('token')[ACTION_UNITS['feature']].mean()
        data_x = dl.get_domain(c_x, s_x).groupby('token')[ACTION_UNITS['feature']].mean()
        a.bar(range(len(tokens)), data_y.corrwith(data_x, axis=1))
        a.axhline(y=.8, c='red')
        if i == 1:
            a.set_xlabel(f"{c_x}+{s_x}")
            a.tick_params(axis='x', rotation=90)
plt.suptitle('Cross-Domain correlations of participants by AU expression')
plt.show()

=> Most participants share their individual AU expressions across domains

In [ ]:
_, axs = plt.subplots(2,2, figsize=(10,6), sharex=True, sharey=True, tight_layout=True)
rows = (('neutral', 'confusion'), ('stress', 'understanding'))
cols = (('neutral', 'understanding'), ('stress', 'confusion'))
for i, ((c_y,s_y), ax) in enumerate(zip(rows, axs)):
    ax[0].set_ylabel(f"{c_y}+{s_y}")
    for (c_x, s_x), a in zip(cols, ax):
        data_y = dl.get_domain(c_y, s_y).groupby('token')[ACTION_UNITS['feature']].mean()
        data_x = dl.get_domain(c_x, s_x).groupby('token')[ACTION_UNITS['feature']].mean()
        a.bar(ACTION_UNITS['name'], data_y.corrwith(data_x))
        if i == 1:
            a.set_xlabel(f"{c_x}+{s_x}")
            a.tick_params(axis='x', rotation=90)
plt.suptitle('Cross-Domain correlations of mean AU expression per participant')
plt.show()

=> Differences in cross domain correlations

### Feature scoring



Prominent AUs for detection understanding/confusion requires low correlation through **state**-domain and high correlation through **condition**-domain

absolute correlation through state change: $c_s = \frac{|corr(x_{nu}, x_{nc})| + |corr(x_{su}, x_{sc})|}{2}$

absolute correlation through condition change: $c_c = \frac{|corr(x_{nu}, x_{su})| + |corr(x_{nc}, x_{sc})|}{2}$

$c_s$ should be small, $c_c$ should be high

Feature score: $s = \frac{1 - c_s + c_c}{2}$

This yields a feature score $s \in [0,1]$ with higher scores representing best features for understanding/confusion detection

In [ ]:
data_nu = dl.get_domain('neutral', 'understanding').groupby('token')[ACTION_UNITS['feature']].mean()
data_su = dl.get_domain('stress', 'understanding').groupby('token')[ACTION_UNITS['feature']].mean()
data_nc = dl.get_domain('neutral', 'confusion').groupby('token')[ACTION_UNITS['feature']].mean()
data_sc = dl.get_domain('stress', 'confusion').groupby('token')[ACTION_UNITS['feature']].mean()

corr_s = (data_nu.corrwith(data_nc).abs() + data_su.corrwith(data_sc).abs()) / 2
corr_c = (data_nu.corrwith(data_su).abs() + data_nc.corrwith(data_sc).abs()) / 2
score = (1 - corr_s + corr_c) / 2

data = pd.concat(
    (
        corr_s.rename('state'),
        corr_c.rename('condition'),
        score.rename('score')
    ), axis=1
)
data.index = ACTION_UNITS['name']
data = data.sort_values('score')
_, axs = plt.subplots(2,2, sharex=True, sharey=True)
ax = data['state'].plot(kind='bar', ax=axs[0,0], title='State change correlation')
ax = data['condition'].plot(kind='bar', ax=axs[0,1], title='Condition change correlation')

ax = data[['state', 'condition']].plot(kind='bar', ax=axs[1,0], title='Comparison')

ax = data['score'].plot(kind='bar', ax=axs[1,1], title='Score')

plt.suptitle('Feature scoring')
plt.show()

print('Top features:')
print('-'*16)
for key in score.sort_values(ascending=False)[:5].index:
    feature_idx = ACTION_UNITS['feature'].index(key)
    print(
        ACTION_UNITS['name'][feature_idx],
        ACTION_UNITS['description'][feature_idx]
    )

=> Top scores for AU26 (Jaw drop), AU45 (Blink), AU9 (Nose Wrinkler), AU02 (Outer Brow Raiser) and AU01 (Inner Brow Raiser)

In [ ]:
ax = data['state'].sort_values().plot(kind='bar', title='State change correlation')

## Frequency exploration

As AU45 (Blinking) truned out to be a prominent feature we can explore blink frequencies through domains

First compute blink durations per video (not per window) to get blink frequnecies

In [ ]:
ACTION_UNIT_BLINK = ACTION_UNITS['feature'][ACTION_UNITS['name'].index('AU45')]
TIME_SECONDS = 'timestamp'

from utils.blink import blink_processor
blink_data = blink_processor(dl.get_full_set(), ACTION_UNIT_BLINK, TIME_SECONDS)
blink_data

Compute frequency (blinks / min): ($60 / duration)$

In [ ]:
blink_data['blink_frequency'] = 60 / blink_data['blink_interval_time']
# mark frequnecies above 100 as invalid
blink_data.loc[blink_data['blink_frequency'] > 100, 'blink_frequency'] = pd.NA

Remove listening state from data

In [ ]:
blink_data_releveant = blink_data[blink_data['state'] != 'listening']

In [ ]:
def boxplot_grouped(data, col_group, col_value, **kwargs):
    plt.boxplot([g.dropna() for _, g in data.groupby(col_group)[col_value]], **kwargs)

boxplot_grouped(
    blink_data_releveant,
    'token', 'blink_frequency', showfliers=False
)
plt.axhline(10, label='10 blinks/min')
plt.legend()
plt.title('Total blink frequencies')
plt.xlabel('participant')
plt.ylabel('blinks/min')
plt.xticks([])
plt.show()

In [ ]:
def plot_freq(ax, condition, state):
    data = blink_data_releveant[
        (blink_data_releveant['condition'] == condition) & 
        (blink_data_releveant['state'] == state)
    ]
    ax.boxplot(
        [g.dropna() for _, g in data.groupby('token')['blink_frequency']],
        showfliers=False
    )
    ax.axhline(10)
    ax.set_xticks([])

plot_cross_domains('In domain blink frequency distribution', plot_freq)

Cross domain comparison

In [ ]:
from scipy import stats
_, axs = plt.subplots(2,2, figsize=(12,6), sharey='row')
# Median
ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['state'] == s
        ].groupby('token')['blink_frequency'].mean().rename(s)
        for s in ('understanding', 'confusion')),
    axis=1
).plot(kind='bar', ax=axs[0,0])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['state'] == 'understanding'
    ].groupby('token')['blink_frequency'].mean(),
    blink_data_releveant[
        blink_data_releveant['state'] == 'confusion'
    ].groupby('token')['blink_frequency'].mean()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])
ax.set_ylabel('Mean')
ax.set_title('State domain')

ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['condition'] == c
        ].groupby('token')['blink_frequency'].mean().rename(c)
        for c in ('neutral', 'stress')),
    axis=1
).plot(kind='bar', ax=axs[0,1])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['condition'] == 'neutral'
    ].groupby('token')['blink_frequency'].mean(),
    blink_data_releveant[
        blink_data_releveant['condition'] == 'stress'
    ].groupby('token')['blink_frequency'].mean()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])
ax.set_title('Condition domain')

# Variance
ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['state'] == s
        ].groupby('token')['blink_frequency'].std().rename(s)
        for s in ('understanding', 'confusion')),
    axis=1
).plot(kind='bar', ax=axs[1,0])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['state'] == 'understanding'
    ].groupby('token')['blink_frequency'].var(),
    blink_data_releveant[
        blink_data_releveant['state'] == 'confusion'
    ].groupby('token')['blink_frequency'].var()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])
ax.set_ylabel('Stadard deviation')

ax = pd.concat((
        blink_data_releveant[
            blink_data_releveant['condition'] == c
        ].groupby('token')['blink_frequency'].std().rename(c)
        for c in ('neutral', 'stress')),
    axis=1
).plot(kind='bar', ax=axs[1,1])
ttest = stats.ttest_rel(
    blink_data_releveant[
        blink_data_releveant['condition'] == 'neutral'
    ].groupby('token')['blink_frequency'].var(),
    blink_data_releveant[
        blink_data_releveant['condition'] == 'stress'
    ].groupby('token')['blink_frequency'].var()
)
ax.set_xlabel(f"p-value = {ttest.pvalue:>.5f}")
ax.set_xticks([])

plt.suptitle('Blink frequnecy change per participant across domains')
plt.show()

=> (Significant?) increase in mean blink frequency per participant for confusion (state-domain)